# Qwen3 Embedding 0.6B INT4 AWQ

This notebook clones `awq-embed`, installs the quantization-only runtime, runs AWQ search on local dummy calibration text, and saves packed INT4 weights for `Qwen/Qwen3-Embedding-0.6B`.

In [ ]:
import torch

print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Enable a GPU runtime before running AWQ quantization.")
print("gpu", torch.cuda.get_device_name(0))
print("bf16 supported", torch.cuda.is_bf16_supported())

In [ ]:
!nvidia-smi

In [ ]:
%cd /content
![ -d awq-embed ] || git clone https://github.com/phunggiahuy159/awq-embed.git
%cd /content/awq-embed

!python -m pip install --upgrade pip
!python -m pip install -e . --no-deps
!python -m pip install "cachetools<7" "transformers>=4.51.0" "accelerate==0.34.2" datasets sentencepiece "tokenizers>=0.12.1" texttable toml attributedict protobuf tqdm

In [ ]:
%%bash
python - <<'PY'
import awq.entry
import awq.quantize.pre_quant
import awq.quantize.auto_scale
import awq.quantize.qmodule
print("AWQ imports succeeded without compiled kernels")
PY

In [ ]:
!python -m awq.entry \
  --model_path Qwen/Qwen3-Embedding-0.6B \
  --model_type embedding \
  --dtype bfloat16 \
  --w_bit 4 \
  --q_group_size 128 \
  --calib_data dummy \
  --calib_n_samples 16 \
  --calib_seqlen 128 \
  --run_awq \
  --dump_awq awq_cache/qwen3-embedding-0.6b-w4-g128-dummy.pt

In [ ]:
!python -m awq.entry \
  --model_path Qwen/Qwen3-Embedding-0.6B \
  --model_type embedding \
  --dtype bfloat16 \
  --w_bit 4 \
  --q_group_size 128 \
  --load_awq awq_cache/qwen3-embedding-0.6b-w4-g128-dummy.pt \
  --q_backend real \
  --dump_quant quant_cache/qwen3-embedding-0.6b-w4-g128-awq.pt

In [ ]:
from pathlib import Path

paths = [
    Path("awq_cache/qwen3-embedding-0.6b-w4-g128-dummy.pt"),
    Path("quant_cache/qwen3-embedding-0.6b-w4-g128-awq-v2.pt"),
]
for path in paths:
    if path.exists():
        print(f"{path}: {path.stat().st_size / (1024 ** 2):.2f} MiB")
    else:
        print(f"missing: {path}")